# Open-Meteo Training Pipeline

Runs the full Open-Meteo branch pipeline: optional weather download, input validation, `main.py` training, and W&B logging.

## 1. Setup

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
PYTHON = sys.executable

def run(cmd, env=None):
    cmd = [str(c) for c in cmd]
    print('$ ' + ' '.join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=env, check=True)

print('repo root:', REPO_ROOT)
print('python   :', PYTHON)

## 2. Configuration

In [ ]:
OPENMETEO_PATH = Path('data/openmeteo_piedmont_2019.nc')
PLANTS_PATH = Path('data/energy_with_coordinates.csv')
SENTINEL_DIR = Path('/data/SentinelPV/energy_data/piemonte_energy_data/single_ups')

SEEDS = [42]                 # one W&B run; add more seeds for multiple runs
BILSTM_POOLING = 'attn'      # 'attn' or 'last'
QS_LOSS_WEIGHTING = '0'
QS_LOSS_FLOOR = '0.2'

WANDB_MODE = 'online'        # use 'offline' only for dry local runs
WANDB_PROJECT = 'PhysiQ-PV'
WANDB_ENTITY = 'albertopedalino-politecnico-di-torino'

RUN_DOWNLOAD_IF_MISSING = True
RUN_PIPELINE_CHECK = True
RUN_TRAINING = True

env = dict(os.environ)
env.update({
    'WEATHER_SOURCE': 'openmeteo_historical_forecast',
    'FEATURE_SET': 'openmeteo_operational',
    'OPENMETEO_PATH': str(OPENMETEO_PATH),
    'SEEDS': ','.join(str(s) for s in SEEDS),
    'BILSTM_POOLING': BILSTM_POOLING,
    'QS_LOSS_WEIGHTING': QS_LOSS_WEIGHTING,
    'QS_LOSS_FLOOR': QS_LOSS_FLOOR,
    'WANDB_MODE': WANDB_MODE,
    'WANDB_PROJECT': WANDB_PROJECT,
    'WANDB_ENTITY': WANDB_ENTITY,
})

print(json.dumps({k: env[k] for k in [
    'WEATHER_SOURCE', 'FEATURE_SET', 'OPENMETEO_PATH', 'SEEDS',
    'BILSTM_POOLING', 'QS_LOSS_WEIGHTING', 'QS_LOSS_FLOOR',
    'WANDB_MODE', 'WANDB_PROJECT', 'WANDB_ENTITY'
]}, indent=2))

## 3. Inputs

In [ ]:
checks = {
    'Sentinel dir': SENTINEL_DIR.exists(),
    'plant coordinates csv': PLANTS_PATH.exists(),
    'Open-Meteo NetCDF': OPENMETEO_PATH.exists(),
}
for name, ok in checks.items():
    print(('OK     ' if ok else 'MISSING') + '  ' + name)

if not SENTINEL_DIR.exists():
    raise FileNotFoundError(f'Sentinel directory not found: {SENTINEL_DIR}')
if not PLANTS_PATH.exists():
    raise FileNotFoundError(f'Plant coordinate file not found: {PLANTS_PATH}')

## 4. Download Open-Meteo If Needed

In [ ]:
if RUN_DOWNLOAD_IF_MISSING and not OPENMETEO_PATH.exists():
    run([
        PYTHON, 'scripts/download_openmeteo_historical_forecast.py',
        '--plants-path', PLANTS_PATH,
        '--start-date', '2019-03-01',
        '--end-date', '2019-12-31',
        '--out', OPENMETEO_PATH,
        '--source', 'historical_forecast',
    ])
else:
    print('Open-Meteo download skipped; file exists or RUN_DOWNLOAD_IF_MISSING=False.')

if not OPENMETEO_PATH.exists():
    raise FileNotFoundError(f'Open-Meteo NetCDF not found: {OPENMETEO_PATH}')

## 5. Pipeline Check

In [ ]:
if RUN_PIPELINE_CHECK:
    run([
        PYTHON, 'scripts/check_openmeteo_pipeline.py',
        '--openmeteo-path', OPENMETEO_PATH,
        '--sentinel-dir', SENTINEL_DIR,
        '--plant-mapping', 'data/plant_mapping.csv',
        '--energy-coords', 'data/energy_with_coordinates.csv',
        '--max-plants', '5',
        '--max-time-steps', '200',
    ])
else:
    print('Pipeline check skipped.')

## 6. W&B

In [ ]:
import wandb
print('wandb version:', wandb.__version__)
print('WANDB_MODE   :', env['WANDB_MODE'])
print('project      :', env['WANDB_PROJECT'])
print('entity       :', env['WANDB_ENTITY'])
print('If this cell fails later at training init, run `wandb login` in the same environment.')

## 7. Train And Log To W&B

In [ ]:
if RUN_TRAINING:
    run([PYTHON, 'main.py'], env=env)
else:
    print('Training skipped. Set RUN_TRAINING=True to execute main.py.')

## 8. Outputs

In [ ]:
checkpoint_dirs = sorted(Path('checkpoints').glob('seq_len_24_pool*seed*'))
print('checkpoint dirs:', len(checkpoint_dirs))
for path in checkpoint_dirs:
    summary = path / 'loss_history.json'
    if summary.exists():
        data = json.loads(summary.read_text())
        best = min(data.get('val', [float('nan')]))
        print(f'{path}  best_val={best:.4f}')
    else:
        print(path)